# fz-CMS Plugin Example Usage

This notebook demonstrates how to use the fz-CMS plugin for parametric CASMO5 / CMSLINK5 / SIMULATE5 calculations.

The examples cover:
1. Installing dependencies (fz framework and fz-CMS plugin)
2. Downloading example input files
3. Parsing input files to identify variables
4. Compiling input files with specific parameter values (variables and formulas)
5. Running CASMO5: pin cell parametric study, assembly depletion
6. Running SIMULATE5: critical boron vs power and inlet temperature
7. Visualization

**Note:** You must have a licensed Studsvik CMS installation, with `CMSHOME` and `LM_LICENSE_FILE` set.

## Setup: Install Dependencies

First, install the Funz framework and the fz-CMS plugin.

In [ ]:
# Install fz framework from GitHub
%pip install -q git+https://github.com/Funz/fz.git

# Install fz-CMS plugin
import fz
fz.install("CMS")

print("✓ Dependencies installed successfully!")

## Setup: Download Example Files

Download the example input files from the GitHub repository.

In [ ]:
import os
import urllib.request

base_url = "https://raw.githubusercontent.com/Funz/fz-CMS/main/examples"
for f in ["CASMO5/pincell.inp", "CASMO5/assembly_dep.inp", "SIMULATE5/quarter_core.inp", "CMSLINK5/library.inp"]:
    os.makedirs(os.path.dirname(f), exist_ok=True)
    urllib.request.urlretrieve(f"{base_url}/{f}", f)
    print(f"✓ Downloaded {f}")

## Setup: Configure the CMS environment

Set `CMSHOME` (directory holding `bin/cas5`, `bin/cmslink5`, `bin/sim5`) and the license server.
`CMS_DATA_PATH` (optional) lists the directories holding the `.lib` / `.cax` files named in the inputs.

In [ ]:
import os

# Adjust for your system
os.environ.setdefault('CMSHOME', '/opt/studsvik/CMSCODES')
os.environ.setdefault('LM_LICENSE_FILE', 'port@license_server')

print(f"CMSHOME set to: {os.environ['CMSHOME']}")

## Import Required Libraries

In [ ]:
import fz
import os
import tempfile

print(f"fz version: {fz.__version__ if hasattr(fz, '__version__') else 'unknown'}")

## Example 1: Parse Input File for Variables

Use `fz.fzi()` to identify variables (and their default values) in a CASMO5 input file.
CASMO5 context lines start with `*@`, SIMULATE5 / CMSLINK5 ones with `'COM'@`.

In [ ]:
variables = fz.fzi("CASMO5/pincell.inp", "CASMO5")
print(f"CASMO5 pin cell variables: {variables}")

variables = fz.fzi("SIMULATE5/quarter_core.inp", "SIMULATE5")
print(f"SIMULATE5 quarter core variables: {variables}")

## Example 2: Compile Input Files

Use `fz.fzc()` to substitute variables and evaluate formulas: the outer radius of the `PIC` card is computed
from the pitch, and the SIMULATE5 inlet temperature is converted from °C to °F.

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    fz.fzc("CASMO5/pincell.inp", {"enrichment": 3.1, "pitch": 1.30}, "CASMO5", output_dir=tmpdir)
    for root, dirs, files in os.walk(tmpdir):
        for f in files:
            if f == "pincell.inp":
                print(open(os.path.join(root, f)).read())

## Example 3: CASMO5 Pin Cell Parametric Study

k-inf of a UO2 pin cell vs enrichment and pitch (moderation ratio). CASMO5 outputs are lists with one value
per state point.

In [ ]:
results = fz.fzr(
    input_path="CASMO5/pincell.inp",
    input_variables={"enrichment": [3.0, 4.0, 5.0], "pitch": [1.20, 1.26, 1.32]},
    model="CASMO5",
    calculators="localhost_CMS",
    results_dir="results_pincell"
)
results["kinf0"] = [k[0] if isinstance(k, list) and k else None for k in results["kinf"]]
print(results[["enrichment", "pitch", "status", "kinf0", "fatal_message"]])

## Example 4: CASMO5 Assembly Depletion

Depletion of a PWR 17x17 assembly for two boron concentrations: `burnup` and `kinf` are the depletion curves.

In [ ]:
dep = fz.fzr(
    input_path="CASMO5/assembly_dep.inp",
    input_variables={"boron": [600, 1000], "max_burnup": 10},
    model="CASMO5",
    calculators="localhost_CMS",
    results_dir="results_dep"
)
print(dep[["boron", "status", "burnup", "kinf"]])

## Example 5: SIMULATE5 Critical Boron

2D quarter core with the library of the SIMULATE5 installation (found through `CMS_DATA_PATH`, by default the
SIMULATE5 / CMSLINK5 `install_inputs` directories).

In [ ]:
s5 = fz.fzr(
    input_path="SIMULATE5/quarter_core.inp",
    input_variables={"power": [0, 50, 100], "tin_c": [286, 291.7]},
    model="SIMULATE5",
    calculators="localhost_CMS",
    results_dir="results_s5"
)
print(s5[["power", "tin_c", "status", "boron", "keff", "fdh"]])

## Example 6: Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for _, row in dep.iterrows():
    if row["status"] == "done":
        ax[0].plot(row["burnup"], row["kinf"], marker=".", label=f"{row['boron']} ppm")
ax[0].set_xlabel("Burnup (MWd/kg)"); ax[0].set_ylabel("k-inf"); ax[0].legend(); ax[0].grid(True)

for tin, grp in s5[s5["status"] == "done"].groupby("tin_c"):
    ax[1].plot(grp["power"], [b[0] for b in grp["boron"]], marker="o", label=f"Tin = {tin} °C")
ax[1].set_xlabel("Power (%)"); ax[1].set_ylabel("Critical boron (ppm)"); ax[1].legend(); ax[1].grid(True)
plt.tight_layout()
plt.show()